# Modelo metiendo las estaciones de fin en el contexto + complejidad

In [14]:
import matplotlib.pyplot as plt

def show_history(history, model_name: str):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10))  # 2 filas, 1 columna

    # Pérdida (loss)
    ax1.plot(history.history['loss'],     label='Training Loss',  color='green')
    ax1.plot(history.history['val_loss'], label='Validation Loss', color='blue')
    ax1.set_title('Loss evolution')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # Precisión (accuracy)
    ax2.plot(history.history['accuracy'],     label='Training Accuracy',  color='green')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', color='blue')
    ax2.set_title('Accuracy evolution')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend()

    # Título global
    fig.suptitle(model_name, fontsize=16)

    # Ajustar márgenes
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [15]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [16]:
with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)
    
with open("../../data/bicycles/df_bicycles_stations.pk1", "rb") as f:
    df_bikesStations = pickle.load(f)

In [ ]:
df_model = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual",
    "month",
    "day",
    "temperature",
    "wind_speed",
    "precipitation",
    "relative_humidity",
    "snow_depth",
    "hour_float"
])

In [18]:
# Variables de entrada
context = df_model.drop(columns=['start_station_idx', 'end_station_idx'])
start = df_model['start_station_idx']
end   = df_model['end_station_idx']

In [19]:
context.head()

,year,event,rideable_type_classic_bike,rideable_type_docked_bike,rideable_type_electric_bike,member_casual_bool,day_type_Holiday,day_type_Normal,day_type_Weekend,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_float,hour_sin,hour_cos,month_sin,month_cos,duration_min
0,2022,False,True,False,False,True,False,True,False,-2.348318,-0.573362,0.916001,-0.099704,-0.056148,4.641111,0.937383,0.348299,0.5,0.866025,13.333333
1,2022,False,True,False,False,True,False,True,False,-2.299739,0.115036,1.211601,-0.099704,-0.056148,11.446944,0.144284,-0.989536,0.5,0.866025,8.683333
2,2022,False,True,False,False,False,False,True,False,-2.299739,0.225945,1.378055,-0.099704,-0.056148,13.231944,-0.316960,-0.948439,0.5,0.866025,38.533333
3,2022,False,False,False,True,True,False,True,False,-1.705422,-0.688095,0.996358,-0.099704,-0.056148,15.498333,-0.793088,-0.609108,0.5,0.866025,6.183333
4,2022,False,True,False,False,True,False,False,True,-2.798966,-1.130964,0.884432,-0.099704,-0.056148,8.467778,0.798460,-0.602047,0.5,0.866025,30.133333


In [20]:
from sklearn.model_selection import train_test_split

ctx_train, ctx__test, start_train, start_test, end_train, end_test = train_test_split(
    context, 
    start, 
    end, 
    test_size=0.2, 
    random_state=42
)

# ---------- Normalizar índices de estaciones con un offset común
station_offset = int(min(start.min(), end.min()))
# desplazamos todos para que el mínimo sea 0
# start_train_shift = (start_train - station_offset).astype(int)
# start_test_shift  = (start_test  - station_offset).astype(int)
# end_train_shift   = (end_train   - station_offset).astype(int)
# end_test_shift    = (end_test    - station_offset).astype(int)

num_stations = int(max(start.max(), end.max()) - station_offset + 1)
num_features = context.shape[1]

print(f"Number of stations: {num_stations}")
print(f"Number of features: {num_features}")
print(f"Station offset: {station_offset}")

# ---------- Preparar inputs en la forma que requieren los Embeddings: (n,1)
input_train_context = ctx_train.values.astype(np.float32)
input_test_context  = ctx__test.values.astype(np.float32)

input_train_start = start_train.values.reshape(-1, 1)
input_test_start  = start_test.values.reshape(-1, 1)

# Labels (etiquetas) para clasificación: estación destino (y_end)
output_train_end = end_train.values.astype(int)
output_test_end  =end_test.values.astype(int)

# ---------- split del train en train/val
input_train_context, input_val_context, input_train_start, input_val_start, output_train_end, output_val_end = train_test_split(
    input_train_context,
    input_train_start,
    output_train_end,
    test_size=0.2,
    random_state=42,
    shuffle=True # No se puede estratificar porque hay estaciones con muy pocos datos
)

Number of stations: 1912
Number of features: 20
Station offset: 0


In [21]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

#embedding_dim = 16  # mucho mejor que sqrt(n)
embedding_dim = int(np.ceil(np.sqrt(num_stations)))

print(f"Embedding dimension: {embedding_dim}")

# Input 1: contexto
context_input = layers.Input(shape=(num_features,), name="context")
# x_ctx = layers.BatchNormalization()(context_input)
# x_ctx = layers.Dense(128, activation="relu")(x_ctx)
# x_ctx = layers.Dropout(0.2)(x_ctx)

# Input 2: estación origen (embedding)
start_input = layers.Input(shape=(1,), name="start_station_input")
start_emb = layers.Embedding(num_stations, embedding_dim)(start_input)
start_emb = layers.Flatten()(start_emb)
# start_emb = layers.Dense(32, activation="relu")(start_emb)

# Concatenación
x = layers.Concatenate()([context_input, start_emb])

x = layers.Dense(254, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)

end_output = layers.Dense(num_stations, activation="softmax")(x)

model = Model(inputs=[context_input, start_input], outputs=end_output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Embedding dimension: 44


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ start_station_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 44)     │     84,128 │ start_station_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 44)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 64)        │          0 │ context[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 254)       │     16,510 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 254)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │     32,640 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1912)      │    124,280 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 265,814 (1.01 MB)

 Trainable params: 265,814 (1.01 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = model.fit(
    {'context': input_train_context, 'start_station_input': input_train_start},
    output_train_end,
    validation_data=(
        {'context': input_val_context, 'start_station_input': input_val_start},
        output_val_end
    ),
    epochs=20,
    batch_size=128,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True # da el mejor modelo
    )]
)


Epoch 1/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 464s 10ms/step - accuracy: 0.0284 - loss: 5.2648 - val_accuracy: 0.0402 - val_loss: 4.9199
Epoch 2/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 360s 8ms/step - accuracy: 0.0423 - loss: 4.8587 - val_accuracy: 0.0488 - val_loss: 4.7510
Epoch 3/20
17635/47554 ━━━━━━━━━━━━━━━━━━━━ 2:55 6ms/step - accuracy: 0.0444 - loss: 4.7943

KeyboardInterrupt: 

# PREDICCIONES

In [ ]:
probs = model.predict({
    "context": X_test_context,
    "start_station_input": start_test_input
})

pred = np.argmax(probs, axis=1) + station_offset
